# Experiment 3 Benchmark: Agentic AI for Outcome-Aware Adaptation

This notebook executes the Experiment 3 benchmark across multiple seeds (Seeds 1, 2, 3). The runner automatically inspects the database and **skips any method/seed that is already completed**, running only the missing experiments.

**Setup Instructions:**
1. Ensure you are using a T4 GPU runtime: `Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU`.
2. Store your GROQ API Key in Google Colab Secrets (Left sidebar -> 🔑 Secrets) with the name `GROQ_API_KEY`.
3. Upload `fl_project_colab.zip` to the Colab files panel.

In [ ]:
# 1. Verify GPU availability (Tesla T4)
!nvidia-smi

In [ ]:
# 2. Install required libraries
!pip install -q torch torchvision numpy pydantic groq python-dotenv tabulate matplotlib

In [ ]:
# 3. Extract package & audit existing completed rounds
import os, zipfile, sqlite3

zip_path = '/content/fl_project_colab.zip'
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('/content')
    print('✅ Extracted fl_project_colab.zip successfully!')
else:
    print('⚠️ fl_project_colab.zip not found at /content/fl_project_colab.zip. Please upload it.')

db_path = 'fl_metrics_experiment3.db' if os.path.exists('fl_metrics_experiment3.db') else 'fl_project/fl_metrics_experiment3.db'
if os.path.exists(db_path):
    con = sqlite3.connect(db_path)
    cur = con.cursor()
    runs = cur.execute('''
        SELECT u.method, u.seed, count(r.id) AS rounds
        FROM experiment3_rounds r
        JOIN experiment3_runs u ON r.run_id = u.run_id
        GROUP BY u.method, u.seed
        ORDER BY u.method, u.seed
    ''').fetchall()
    print('\n--- Currently Completed Runs in Database ---')
    for method, seed, rounds in runs:
        print(f'  - {method:<18} (Seed {seed}): {rounds}/20 rounds [Complete]')
    con.close()

In [ ]:
# 4. Mount Google Drive & load GROQ API Key
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

try:
    os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
    print('✅ GROQ_API_KEY loaded successfully from Colab Secrets.')
except Exception as e:
    print('⚠️ Please store your key in Colab Secrets with the name GROQ_API_KEY.')

In [ ]:
# 5. Fast CIFAR-10 download (5 seconds)
!apt-get install -qq aria2 && aria2c -x 16 -s 16 -d ./data -o cifar-10-python.tar.gz https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz

In [ ]:
# ==========================================================================
# STEP A (PRIMARY GOAL): Run AgenticAI (Seed 1)
# Completes the final method so all 8 canonical methods are done for Seed 1
# Duration: ~10-12 minutes
# ==========================================================================
!python fl_project/run_experiment3.py --methods AgenticAI --seeds 1

In [ ]:
# ==========================================================================
# STEP B: Complete All Baselines across Seeds 1, 2, 3
# Automatically skips completed ones (FixedFedAvg s1/s2, FixedMedian s1/s2, etc.)
# Zero LLM calls required — completes in ~8-10 minutes on T4 GPU
# ==========================================================================
!python fl_project/run_experiment3.py --methods FixedFedAvg FixedMedian FixedTrimmedMean FixedKrum RuleBased --seeds 1 2 3

In [ ]:
# ==========================================================================
# STEP C (MULTI-SEED EXTENSION): Run LLM Methods for Seeds 2 & 3
# Runs SingleShotLLM, ReflectiveAgent, AgenticAI for Seeds 2 & 3
# ==========================================================================
!python fl_project/run_experiment3.py --methods SingleShotLLM ReflectiveAgent AgenticAI --seeds 2 3

In [ ]:
# 6. Automatic Permanent Backup to Google Drive
import shutil, os

dst_dir = '/content/drive/MyDrive/fl_experiment3_backup'
os.makedirs(dst_dir, exist_ok=True)

for cand in ['fl_metrics_experiment3.db', 'fl_project/fl_metrics_experiment3.db']:
    if os.path.exists(cand):
        dst = os.path.join(dst_dir, 'fl_metrics_experiment3.db')
        shutil.copy2(cand, dst)
        print(f'✅ Backed up database to Google Drive: {dst} ({os.path.getsize(dst):,} bytes)')
        break

In [ ]:
# 7. Generate Complete Summary Tables (All Methods & Seeds)
import sqlite3, pandas as pd, numpy as np

db_path = 'fl_metrics_experiment3.db' if os.path.exists('fl_metrics_experiment3.db') else 'fl_project/fl_metrics_experiment3.db'
con = sqlite3.connect(db_path)

query = '''
    SELECT 
        u.method,
        u.seed,
        r.round,
        r.alpha,
        r.attack_active,
        r.aggregation_method,
        r.test_accuracy,
        r.test_f1,
        r.oracle_accuracy,
        r.regret,
        r.cumulative_regret,
        r.round_time_s
    FROM experiment3_rounds r
    JOIN experiment3_runs u ON r.run_id = u.run_id
    ORDER BY u.method, u.seed, r.round
'''
df = pd.read_sql_query(query, con)
con.close()

print(f'Total evaluated rounds across all runs: {len(df)}\n')

records = []
for (method, seed), grp in df.groupby(['method', 'seed']):
    mean_acc = grp['test_accuracy'].mean()
    final_acc = grp.iloc[-1]['test_accuracy']
    attack_grp = grp[grp['attack_active'] == 1]
    attack_acc = attack_grp['test_accuracy'].mean() if len(attack_grp) > 0 else np.nan
    final_cum_regret = grp.iloc[-1]['cumulative_regret']
    mean_regret = grp['regret'].mean()
    mean_time = grp['round_time_s'].mean()
    
    records.append({
        'Method': method,
        'Seed': seed,
        'Rounds': len(grp),
        'Mean Acc (%)': round(mean_acc, 2),
        'Final Acc (%)': round(final_acc, 2),
        'Byzantine Acc (R16-20) (%)': round(attack_acc, 2) if not np.isnan(attack_acc) else 'N/A',
        'Mean Regret (%)': round(mean_regret, 2),
        'Cumul Regret (%)': round(final_cum_regret, 2),
        'Avg Time/Rnd (s)': round(mean_time, 2)
    })

summary_df = pd.DataFrame(records)
print('=' * 95)
print('EXPERIMENT 3 BENCHMARK: INDIVIDUAL RUN BREAKDOWN')
print('=' * 95)
display(summary_df)

print('\n' + '=' * 95)
print('AGGREGATE METRICS ACROSS SEEDS (FOR PUBLICATION TABLES)')
print('=' * 95)
agg_list = []
for method, grp in summary_df.groupby('Method'):
    n = len(grp)
    mean_acc = f"{grp['Mean Acc (%)'].mean():.2f} ± {grp['Mean Acc (%)'].std():.2f}" if n > 1 else f"{grp['Mean Acc (%)'].mean():.2f}"
    final_acc = f"{grp['Final Acc (%)'].mean():.2f} ± {grp['Final Acc (%)'].std():.2f}" if n > 1 else f"{grp['Final Acc (%)'].mean():.2f}"
    cum_reg = f"{grp['Cumul Regret (%)'].mean():.2f} ± {grp['Cumul Regret (%)'].std():.2f}" if n > 1 else f"{grp['Cumul Regret (%)'].mean():.2f}"
    byz_vals = grp[grp['Byzantine Acc (R16-20) (%)'] != 'N/A']['Byzantine Acc (R16-20) (%)'].astype(float)
    byz_acc = f"{byz_vals.mean():.2f}" if len(byz_vals) > 0 else 'N/A'
    
    agg_list.append({
        'Method': method,
        'Seeds Tested': n,
        'Mean Accuracy (%)': mean_acc,
        'Final Accuracy (%)': final_acc,
        'Byzantine Defense Acc (%)': byz_acc,
        'Cumulative Regret (%)': cum_reg
    })

display(pd.DataFrame(agg_list))

In [ ]:
# 8. Generate Publication Figure (Rounds 1–20 Trajectories)
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

for method, grp in df[df['seed'] == 1].groupby('method'):
    plt.plot(grp['round'], grp['test_accuracy'], marker='o', label=method, linewidth=2.0)

plt.axvspan(15.5, 20.5, color='red', alpha=0.12, label='Byzantine Attack Window (Rounds 16–20)')

plt.title('CIFAR-10 FL: Outcome-Aware Adaptation under Dynamic Dirichlet & Byzantine Attacks', fontsize=13, fontweight='bold')
plt.xlabel('Communication Round', fontsize=11)
plt.ylabel('Test Accuracy (%)', fontsize=11)
plt.ylim(50, 100)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(bbox_to_anchor=(1.04, 1), loc='upper left')
plt.tight_layout()
plt.savefig('experiment3_byzantine_defense.png', dpi=300)
plt.show()
print('✅ Saved publication figure to experiment3_byzantine_defense.png')